In [ ]:
#import libraries
import grama as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
#set DF as the dataframe pronoun
DF = gr.Intention()

In [ ]:
#constants
free_space_permeability = 4*np.pi*10**-7
relativity_permeability_of_air = 1.0000037
effective_permeability = relativity_permeability_of_air/free_space_permeability

#set up gramma model inputs and outputs
var_electro = ["winds","current","eff_core_length","core_width"]
out_electro = ["magnetic_field"]

#define a function for calculating the electromagnetic field strength
def electromatnetic_field(df):

    #equation for the electromagnetic field strength
    magnetic_field = (effective_permeability*df.winds*df.current)/2 * ...
    (np.cos(np.tan(df.core_width/(df.eff_core_length+df.magnet_dist))) - np.cos(np.tan(df.core_width/df.magnet_dist)))

    #convert field strength to force
        #magnetic_force = (magnetic_field**2 * overlapping_area)/(2*free_space_permeability)

    return gr.df_make(field=magnetic_field)

Model rational:
- Assume: We care about maximizing force
   - This may not be true, maybe we have a target force, but for now let's say this is true
- Assume: This is a single objective optimization problem
   - This essentially encodes the fact that we care about force the most and are willing to make compromises on other objectives such as winds, current, core length, etc.
   - This may be true, and is a useful assumption for even seeing if we are in the right ballpark of forces

Therefore, the optimization problem is as follows:
$$max$$
$$force$$
$$wrt$$
$$winds, current, core length$$
$$s.t. $$
$$winds <= C_1*core length$$
$$current <= C_2$$
$$core length <= C_3$$
Where $C_1$ is the maximum number of winds per unit core length, $C_2$ is our maximum reasonable current draw, and $C_3$ is the maximum reasonable core length

If we actually care about electrical or geometric constraints and we have a known requisite force, the optimization problem will look like one of the following:
$$\min\, current$$
$$\text{wrt.}\, winds, core length$$
$$\text{s.t.}\, winds <= C_1*core length$$
$$\text{s.t.}\, core length <= C_2$$
$$\text{s.t.}\, force >= C_3$$
Where: 
- $C_1$ is the maximum number of winds per unit core length
- $C_2$ is the maximum reasonable core length
- $C_3$ is the minimum necessary force

$$\min\, f(x)$$
$$\text{wrt.}\, x$$
$$\text{s.t.}\, u(x) \leq 0$$
$$\text{s.t.}\, v(x) = 0$$

or:
$$min$$
$$core length$$
$$wrt$$
$$winds, current$$
$$s.t.$$
$$winds <= C_1*core length$$
$$current <= C_2$$
$$force >= C_4$$

In [ ]:
#create gramma model
md_electro =(
    gr.Model()
    #add field strength function
    >> gr.cp_vec_function(
        fun=electromatnetic_field,
        var=var_electro,
        out=out_electro
    )
    #add function bounds
    >> gr.cp_bounds(
        winds=(),
        current=(),
        eff_core_length=(),
        core_width=(),
        magnet_dist=(),
        overlapping_area=()
    )
)

IndexError: tuple index out of range

In [ ]:
#run the minimizer
df_res = (
    md_electro
    >> gr.ev_min(
        out_min=out_electro,
        n_restarts=10
    )
)

#print out the results
md_electro.printpretty()

$$\min\, f(x)$$
$$\text{wrt.}\, x$$
$$\text{s.t.}\, u(x) \leq 0$$
$$\text{s.t.}\, v(x) = 0$$